# Tutorial: write `helpers.py` (no pandas)

Project 1 forbids pandas. The course still wants two helpers in a file at the **repo root** called `helpers.py`:

1. `load_csv_data` — read `x_train.csv`, `y_train.csv`, `x_test.csv`
2. `create_csv_submission` — write `Id,Prediction` like `sample_submission.csv`

This notebook is a **learning notes** file. Run it cell by cell, then copy the last two functions into `/ML_project_1/helpers.py`.

Allowed: Python standard library (`os`, `csv`) + NumPy. Not allowed: `import pandas`.

In [1]:
import os
import csv
import numpy as np

# Notebook lives in doc/, data lives in ../data/
DATA_DIR = os.path.join("..", "data")
print("files:", os.listdir(DATA_DIR))

files: ['README.md', 'x_train.csv', 'y_train.csv', 'x_test.csv', 'sample_submission.csv']


## 1. What the CSV files look like

| File | First column | Rest |
|---|---|---|
| `x_train.csv` | `Id` | 321 features |
| `y_train.csv` | `Id` | `_MICHD` in `{+1, -1}` |
| `x_test.csv` | `Id` | same 321 features, **no label** |
| `sample_submission.csv` | `Id` | `Prediction` in `{+1, -1}` |

Row 0 of the file is a **header** (column names). Data starts at row 1.

Many survey cells are **empty**. NumPy should turn those into `nan` (not a number), not crash.

Keep the three names unchanged, in the same folder. That is what the project PDF asks.

## 2. `np.genfromtxt` — read a table without pandas

`np.genfromtxt(path, ...)` reads a text file into an array.

| Argument | Why we use it |
|---|---|
| `delimiter=","` | columns are separated by commas |
| `skip_header=1` | skip the name row |
| `filling_values=np.nan` | empty cell → `nan` |
| `max_rows=5` | **only for this tutorial** so we do not load 300k rows in every cell |

In the real `helpers.py`, **do not** set `max_rows` (load everything).

In [2]:
y_raw = np.genfromtxt(
    os.path.join(DATA_DIR, "y_train.csv"),
    delimiter=",",
    skip_header=1,
    max_rows=5,
)
print("y_raw shape", y_raw.shape)
print(y_raw)
print("column 0 = Id, column 1 = label")

y_raw shape (5, 2)
[[ 0. -1.]
 [ 1. -1.]
 [ 2. -1.]
 [ 3. -1.]
 [ 4. -1.]]
column 0 = Id, column 1 = label


Split that table:

- `[:, 0]` = all rows, first column → ids
- `[:, 1]` = all rows, second column → labels

`astype(int)` turns `0.0` into `0`, `-1.0` into `-1`.

In [3]:
ids_y = y_raw[:, 0].astype(int)
labels = y_raw[:, 1].astype(int)
print("ids   ", ids_y)
print("labels", labels)

ids    [0 1 2 3 4]
labels [-1 -1 -1 -1 -1]


Now features. `x_train` has many columns, so we only print **shape** and the first few ids.

In [4]:
x_raw = np.genfromtxt(
    os.path.join(DATA_DIR, "x_train.csv"),
    delimiter=",",
    skip_header=1,
    filling_values=np.nan,
    max_rows=5,
)
print("x_raw shape", x_raw.shape, "  (5 rows × 1 id + 321 features)")
ids_x = x_raw[:, 0].astype(np.int64)
features = x_raw[:, 1:]
print("ids     ", ids_x)
print("features", features.shape)
print("nan count in these 5 rows", np.isnan(features).sum())

x_raw shape (5, 322)   (5 rows × 1 id + 321 features)
ids      [0 1 2 3 4]
features (5, 321)
nan count in these 5 rows 732


`features` is 2D: rows = people, columns = measurements. That is the `tx` you pass into `least_squares` later.

Train ids in `x_train` and `y_train` should match. If they do not, you joined the wrong rows.

## 3. `load_csv_data`

The PDF says: keep the three csv files in **one folder**, do not rename them, and load with `load_csv_data`.

A useful return value (what we will put in `helpers.py`):

| Name | Shape | Meaning |
|---|---|---|
| `y_train` | `(N,)` | `{+1, -1}` |
| `x_train` | `(N, D)` | features |
| `ids_train` | `(N,)` | train ids |
| `x_test` | `(N_test, D)` | test features |
| `ids_test` | `(N_test,)` | test ids (must go into the submission) |

`sub_sample=True` is optional: take every 50th row so EDA is faster. Default `False` for the real run.

In [5]:
def load_csv_data(data_path, sub_sample=False):
    """Load Project 1 csv files from one folder.

    Expects x_train.csv, y_train.csv, x_test.csv in data_path.

    Returns:
        y_train, x_train, ids_train, x_test, ids_test
    """
    y_file = np.genfromtxt(
        os.path.join(data_path, "y_train.csv"),
        delimiter=",",
        skip_header=1,
    )
    y_train = y_file[:, 1].astype(int)
    ids_from_y = y_file[:, 0].astype(np.int64)

    x_file = np.genfromtxt(
        os.path.join(data_path, "x_train.csv"),
        delimiter=",",
        skip_header=1,
        filling_values=np.nan,
    )
    ids_train = x_file[:, 0].astype(np.int64)
    x_train = x_file[:, 1:]

    if not np.array_equal(ids_train, ids_from_y):
        raise ValueError("Ids in x_train.csv and y_train.csv do not match.")

    x_test_file = np.genfromtxt(
        os.path.join(data_path, "x_test.csv"),
        delimiter=",",
        skip_header=1,
        filling_values=np.nan,
    )
    ids_test = x_test_file[:, 0].astype(np.int64)
    x_test = x_test_file[:, 1:]

    if sub_sample:
        y_train = y_train[::50]
        x_train = x_train[::50]
        ids_train = ids_train[::50]

    return y_train, x_train, ids_train, x_test, ids_test


print("function defined. Next cell loads the full files (can take ~30s).")

function defined. Next cell loads the full files (can take ~30s).


The next cell reads **all** rows. Run it once. If it is too slow, set `sub_sample=True`.

In [6]:
y_train, x_train, ids_train, x_test, ids_test = load_csv_data(
    DATA_DIR, sub_sample=True
)
print("y_train", y_train.shape, "unique", np.unique(y_train))
print("x_train", x_train.shape)
print("ids_train", ids_train[:5])
print("x_test", x_test.shape)
print("ids_test", ids_test[:5])
print("fraction of nan in x_train", np.mean(np.isnan(x_train)))

KeyboardInterrupt: 

## 4. `create_csv_submission`

AIcrowd wants **exactly**:

```text
Id,Prediction
328135,-1
328136,1
```

- First row is the header `Id,Prediction`.
- `Id` must be the **test** ids (`ids_test`), same order as `x_test.csv`.
- `Prediction` must be `+1` or `-1` (integers), not `0/1` and not probabilities.

We write with the standard library `csv` module (allowed).

In [7]:
def create_csv_submission(ids, y_pred, name):
    """Write a submission file.

    Args:
        ids: 1D array of test ids
        y_pred: 1D array of predictions in {+1, -1}
        name: output path, e.g. 'submission.csv'
    """
    with open(name, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Id", "Prediction"])
        for i, pred in zip(ids, y_pred):
            writer.writerow([int(i), int(pred)])


# Tiny demo: pretend the first 3 test people are all healthy (-1)
demo_path = os.path.join(DATA_DIR, "_demo_submission.csv")
create_csv_submission(ids_test[:3], np.array([-1, -1, -1]), demo_path)
with open(demo_path) as f:
    print(f.read())

NameError: name 'ids_test' is not defined

After you have a real model, typical use:

```python
y_hat = np.sign(x_test @ w)          # least squares → {+1, -1, 0}
y_hat[y_hat == 0] = -1               # break the rare 0 case
create_csv_submission(ids_test, y_hat, "submission.csv")
```

For logistic, labels were `{0,1}` during training, so map back: `2 * y01 - 1`.

## 5. Copy into `helpers.py`

Create `/ML_project_1/helpers.py` with:

- `import os`, `import csv`, `import numpy as np`
- the two functions above (without `max_rows`)

Then from `run.py` (later):

```python
from helpers import load_csv_data, create_csv_submission

y, x, ids_tr, x_te, ids_te = load_csv_data("data")
```

Paths: if you run `python run.py` from the **repo root**, `data_path="data"`. This notebook uses `"../data"` because it lives in `doc/`.

**Do not** commit the big csv files. `create_csv_submission` output can stay local too until you upload it to AIcrowd.